# MultimodalVA — Looking at the Results

You have trained something. This notebook is about the other half of the work:
**what did it predict, how good is it, and where does it fail?**

Read `01_start_here_text_and_tabular.ipynb` first if you have not — this one
assumes you can already get a result out of `mmva.run()`.

Covered here:

1. Training **several models at once**, and comparing them fairly
2. The **predicted cause distribution** — the one thing you can see without any true labels
3. A **leaderboard** across models
4. **Confidence intervals**, and whether two models really differ
5. **Confusion matrices** and per-cause accuracy — where it goes wrong
6. **CSMF** — getting the population right rather than the individual
7. **Top-k** accuracy — is the right cause at least in the top 3?
8. **Calibration** — are the probabilities honest?
9. **Search diagnostics** — what the hyperparameter search did
10. A **full menu** of everything in `multimodalva.results`

Every section is independent. Skip to whichever one you need.

## 1. Training several models at once

You are not limited to one model per run. There are two ways to train several,
and the difference between them matters.

### Way 1 — a loop

The plainest approach. Each model is trained independently, and you get one
result per model.

The one thing to be careful about: **every model must see the same train/test
split**, or the comparison is meaningless. Passing the same `split_seed` and
the same DataFrame gives every model the same split. For anything more serious,
fix the split explicitly with `split=` so it survives across sessions and
machines.

In [ ]:
import multimodalva as mmva
import pandas as pd

df = mmva.data("va_sample", n_per_class=40)
LABEL, TEXT = "cause_of_death", "narrative"
features = [c for c in df.columns if c.startswith("i") and c[1:4].isdigit()]

results = {}
for name in ["lightgbm", "random_forest", "naive_bayes"]:
    results[name] = mmva.run(
        task="tabular",
        data=df,
        label_col=LABEL,
        features=features,
        model=name,
        output_dir=f"runs/compare/{name}",
        split_seed=42,            # same split_seed -> same split for every model
    )

print("trained:", list(results))

You can mix pipelines in the same loop — a `text` model and a `tabular` model
are directly comparable as long as they share the split:

```python
results["bluebert"] = mmva.run(task="text", data=df, label_col=LABEL,
                               text_col=TEXT, model="bluebert",
                               output_dir="runs/compare/bluebert",
                               split_seed=42)
```

### Way 2 — one call, with `task="voting"`

If you were going to combine the models anyway, `voting` trains them all in a
single call. It splits the data **once** and shares that split with every base
model, so the comparison is automatically fair, and each model can carry its own
settings.

Each base model is saved separately under `base_models/`, so you can evaluate
them individually as well as the ensemble.

In [ ]:
vote = mmva.run(
    task="voting",
    data=df,
    label_col=LABEL,
    text_col=TEXT,
    features=features,
    output_dir="runs/compare/voting",
    # Each dict is one model, with its own settings. Add as many as you like.
    tabular_models=[
        {"model_name": "lightgbm"},
        {"model_name": "random_forest"},
        {"model_name": "naive_bayes"},
    ],
    # Text models go in their own list, and can each be tuned separately:
    # text_models=[
    #     {"model_name": "bluebert", "max_length": 512,
    #      "hyperparams": mmva.Optimize(n_trials=20)},
    # ],
)

# The ensemble's own predictions, plus one PredictionResult per base model:
print("base models trained:", len(vote["base_predictions"]))

Note that `voting` and `stacking` write into a named sub-folder of your
`output_dir` — `runs/compare/voting/soft_voting/` here, and `.../stacking/` for
stacking. The base models are under `base_models/text_<i>/` and
`base_models/tabular_<i>/`, numbered in the order you listed them.

`stacking` works the same way and takes the same two lists, but it also trains a
model to combine the others, rather than averaging them.

## 2. What did the model predict?

This is the only view that needs **no true labels at all**, so it is also what
you use on data where the cause of death is unknown — which is the real use case.

It is plain pandas; no special function required.

In [ ]:
import matplotlib.pyplot as plt

pred = results["lightgbm"]["predictions"].top1   # columns: id, true_label, predicted_label, ...

counts = pred["predicted_label"].value_counts()

ax = counts.sort_values().plot.barh(figsize=(7, 5), color="#00468B")
ax.set_xlabel("deaths assigned")
ax.set_title("Predicted cause-of-death distribution — lightgbm")
plt.tight_layout()
plt.show()

In [ ]:
# As fractions — this is the predicted CSMF (cause-specific mortality fraction),
# the number most VA studies actually report.
csmf = (counts / counts.sum()).rename("predicted_fraction")
csmf.head(10)

Each model also gives you the full probability table and the ranked top-k, not
just the single best guess:

```python
result["predictions"].top1   # one row per death: the single predicted cause
result["predictions"].full   # one column per cause: the probability of each
result["predictions"].topk   # the k most likely causes, ranked, with probabilities
result["predictions"].id2label   # which column is which cause
```

## 3. A leaderboard across models

`predictions_frame()` gathers several models into one table, and
`performance_leaderboard()` scores them all against the true labels.

`predictions_frame()` is deliberately undemanding about what you hand it. Each
value can be a `run()` result, a `PredictionResult`, a bare `top1` DataFrame, or
**just the path to a finished run directory** — so you can build a leaderboard
weeks later from runs on disk, without reloading a single model.

In [ ]:
from multimodalva.results import predictions_frame, performance_leaderboard

frame = predictions_frame(results)
frame.head(3)

In [ ]:
board = performance_leaderboard(frame, "true_label")
board

The same thing, from directories — nothing but paths:

In [ ]:
frame_from_disk = predictions_frame({
    "lightgbm":      "runs/compare/lightgbm",
    "random_forest": "runs/compare/random_forest",
    "voting":        "runs/compare/voting/soft_voting",
})
performance_leaderboard(frame_from_disk, "true_label")

If two models were trained on different splits, `predictions_frame()` refuses to
build the table rather than silently comparing different sets of deaths:

> `Model 'x' has 88 test cases but 'y' has 66. All models must be evaluated on
> the same test set — re-run them with the same split.`

### More metrics

By default the leaderboard reports the ten label-based metrics. Two extras need
more than the predicted label, so you pass the probability and top-k tables:
`log_loss` (are the probabilities well calibrated?) and `topN_accuracy`.

In [ ]:
from multimodalva.utils.metrics import REPORT_METRICS

board_full = performance_leaderboard(
    frame, "true_label",
    metrics=[*REPORT_METRICS, "log_loss"],
    prob_dfs={n: r["predictions"].full for n, r in results.items()},
    id2label=results["lightgbm"]["predictions"].id2label,
    topk_dfs={n: r["predictions"].topk for n, r in results.items()},
    top_k=3,
)
board_full

**What the metrics mean, briefly.** `accuracy` is the share of deaths assigned
the right cause. `balanced_accuracy` and `f1_macro` weigh every cause equally,
so a model cannot look good by only getting the common causes right.
`csmf_accuracy` scores the *distribution* of causes across the population rather
than individual deaths — it is the standard VA measure, and a model can score
well on it while getting many individuals wrong. `cccsmf_accuracy` is the
chance-corrected version.

## 4. Confidence intervals

A single number hides how much of it is luck. `bootstrap_ci()` resamples the
test cases and reports the interval. All models are scored on the **same**
resamples, so the intervals are comparable.

In [ ]:
from multimodalva.results import bootstrap_ci

bootstrap_ci(results,
             metrics=["accuracy", "f1_macro", "csmf_accuracy"],
             n_boot=1000,
             formatted=True)     # ready-to-paste "95.5 (90.9-98.9)" cells

Overlapping intervals do **not** mean two models are indistinguishable. To ask
whether A really beats B, bootstrap the *difference*: if the interval for the
difference excludes zero, the gap is unlikely to be noise.

In [ ]:
from multimodalva.results import paired_bootstrap_ci

paired_bootstrap_ci(results,
                    reference="lightgbm",       # everything is compared against this
                    metrics=["accuracy", "f1_macro"],
                    n_boot=1000,
                    formatted=True)

## 5. Where does it go wrong?

A confusion matrix shows which causes get mistaken for which. `normalize=True`
makes each row sum to 1, so rare causes are readable next to common ones.

In [ ]:
from multimodalva.results import confusion_heatmap

top1 = results["lightgbm"]["predictions"].top1
confusion_heatmap(
    top1["true_label"], top1["predicted_label"],
    model_name="lightgbm",
    normalize=True,
    figsize=(9, 7),
)
plt.show()

A cause-accuracy heatmap is the multi-model version: one row per cause, one
column per model, so you can see which model handles which cause.

In [ ]:
from multimodalva.results import cause_accuracy_heatmap

cause_accuracy_heatmap(frame, "true_label", figsize=(9, 4.5))
plt.show()

And the difference against a baseline — red/blue for worse/better — which is
usually the more readable figure when you are arguing that one model improved on
another.

In [ ]:
from multimodalva.results import cause_accuracy_diff_heatmap

cause_accuracy_diff_heatmap(
    frame, "true_label",
    baseline_col="lightgbm",     # every other column is shown relative to this
    figsize=(9, 4),
)
plt.show()

Both heatmaps also accept `top_k=` with the top-k tables, to ask "was the right
cause in the top 3?" per cause rather than "was it first?".

Useful options on both: `drop_causes=` to remove causes, `cause_rename=` /
`model_rename=` for publication labels, `group_sizes=` with `group_labels=` to
draw boxes around blocks of related causes, and `save_path=` to write the figure
straight to a file.

In [ ]:
cause_accuracy_heatmap(
    frame, "true_label",
    top_k=3,
    topk_dfs={n: r["predictions"].topk for n, r in results.items()},
    figsize=(9, 4.5),
)
plt.show()

## 6. Getting the population right — CSMF

For most VA work the question is not "was this death classified correctly?" but
"is the cause distribution of this population right?". A model can be mediocre
at individuals and still estimate the population well, because its errors cancel.

Each point is one cause: the true share against the predicted share. Points on
the diagonal are right; above it the model over-assigns that cause, below it
under-assigns.

In [ ]:
from multimodalva.results import csmf_scatterplot

csmf_scatterplot(
    top1["true_label"], top1["predicted_label"],
    title="CSMF — lightgbm",
)
plt.show()

## 7. Was the right cause at least in the top 3?

VA models are often used to narrow a list rather than to give one answer. Top-k
accuracy asks whether the true cause appeared in the k most likely.

In [ ]:
from multimodalva.results import topk_accuracy

topk_accuracy(results["lightgbm"]["predictions"].topk)

In [ ]:
from multimodalva.results import plot_topk_accuracy

plot_topk_accuracy({n: r["predictions"].topk for n, r in results.items()})
plt.show()

## 8. Are the probabilities honest?

A model that says "80% Malaria" should be right about 80% of the time when it
says that. When it is not, the model is **miscalibrated**, and its probabilities
should not be used as confidence — which matters if anyone downstream thresholds
on them.

- **ECE** — average gap between stated confidence and actual accuracy (lower is better)
- **MCE** — the single worst gap
- **Brier** — overall probability error, combining calibration and accuracy

In [ ]:
from multimodalva.results import calibration_summary

for name, r in results.items():
    s = calibration_summary(r["predictions"].full, r["predictions"].id2label)
    print(f"{name:16s}", {k: round(v, 4) for k, v in s.items()})

In [ ]:
# With a bootstrap confidence interval (slower):
calibration_summary(
    results["lightgbm"]["predictions"].full,
    results["lightgbm"]["predictions"].id2label,
    n_boot=500,
)

## 9. What did the hyperparameter search do?

When a run used `hyperparams=Optimize(...)`, its `hpo/` folder holds every trial. This
tells you whether the search had converged or was still improving when it
stopped — i.e. whether more trials would have been worth it.

In [ ]:
hpo_run = mmva.run(
    task="tabular", data=df, label_col=LABEL, features=features,
    model="lightgbm", output_dir="runs/compare/lightgbm_hpo",
    hyperparams=mmva.Optimize(n_trials=8, metric="f1_macro"),
    split_seed=42,
)

In [ ]:
from multimodalva.results import hpo_leaderboard

trials = pd.read_csv("runs/compare/lightgbm_hpo/hpo/hpo_trials.csv")
hpo_leaderboard(trials, top_n=5)

Every HPO run also writes `hpo/hpo_convergence.png` and `hpo/hpo_leaderboard.csv`
automatically, so you do not have to produce these by hand.

If you still have the Optuna study object, `hpo_convergence_plot(study)`,
`hyperparameter_importance(study)` and `plot_param_importances(study)`
(interactive Plotly) all take it directly.

## 10. The full menu

Everything below is importable from `multimodalva.results`. Pick what you need.

**Comparing models**

| Function | Gives you |
|---|---|
| `predictions_frame` | Gathers several models into one table — accepts results, `PredictionResult`s, DataFrames or run directories |
| `performance_leaderboard` | All ten metrics per model, ranked |
| `bootstrap_ci` | Confidence interval per model per metric |
| `paired_bootstrap_ci` | Confidence interval on the *difference* between models |

**Where the errors are**

| Function | Gives you |
|---|---|
| `confusion_heatmap` | Which causes get confused with which, one model |
| `cause_accuracy_heatmap` | Per-cause accuracy, several models side by side |
| `cause_accuracy_diff_heatmap` | Per-cause change against a baseline model |
| `csmf_scatterplot` | True vs predicted share of each cause |

**Ranked predictions**

| Function | Gives you |
|---|---|
| `topk_accuracy` | Accuracy at each k, as a table |
| `plot_topk_accuracy` | The same as a bar chart, one or many models |
| `topk_from_full` | Builds a top-k table from a probability table |

**Probability quality**

| Function | Gives you |
|---|---|
| `calibration_summary` | ECE, MCE and Brier together, optionally with a CI |
| `ece_score`, `mce_score`, `brier_multiclass` | The individual scores |
| `classwise_bin_data` | The underlying per-cause bins, to plot yourself |

**Training and search diagnostics**

| Function | Gives you |
|---|---|
| `hpo_leaderboard` | Trials ranked, with their hyperparameters |
| `hpo_convergence_plot` | Did the search converge? |
| `hpo_loss_trend`, `hpo_metric_variance` | How the objective and metrics moved across trials |
| `hyperparameter_importance`, `plot_param_importances` | Which hyperparameters mattered |
| `plot_loss_curves`, `loss_curve_diagnostics`, `train_eval_gap` | Training curves and overfitting, for text models |
| `oov_rate` | How much of your text the tokenizer does not know — a check on whether the checkpoint suits your data |

**Colour palettes** — `HEATMAP_CLINICAL`, `HEATMAP_SEQ`, `HEATMAP_DIV`,
`TOPK_BAR_COLORS`, passed as `cmap=` / `colors=`.

Most plotting functions take `save_path=` and `dpi=` to write a figure directly
to a file, and return their matplotlib objects so you can adjust anything before
saving.

---

**Next:** to run any of this outside a notebook, copy
`04_python_script_template.py`, `05_command_line_template.sh`, or
`06_config_file_template.sh` with one of the `config_*.yaml` files.